# Multimodal Topic Mining and Image Classification
## Advanced Customer Analytics - Visual Data Predictions Assignment


In [7]:
# Install required packages
# !pip install -q bertopic
# !pip install -q scikit-learn
# !pip install -q pillow
# !pip install -q pandas
# !pip install -q matplotlib
# !pip install -q seaborn
# !pip install -q requests
# !pip install -q tqdm
# !pip install -q datasets
# !pip install -q kaggle

In [8]:
import os
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import zipfile
import subprocess


### Download the dataset and the images

In [9]:
def download_and_process_pokemon_cards(save_dir='pokemon_cards', num_cards=1000):
    """
    Download Pokemon TCG dataset from Kaggle and download card images.
    Everything is saved in a single directory.
    
    Args:
        save_dir: Directory to save everything (images and metadata)
        num_cards: Number of cards to process
        
    Returns:
        df_final: Processed dataframe with image paths
    """
    
    # Create directory structure
    os.makedirs(f"{save_dir}/images", exist_ok=True)
    temp_dir = f"{save_dir}/temp"
    os.makedirs(temp_dir, exist_ok=True)
    
    # Step 1: Download dataset from Kaggle
    print("Downloading Pokemon TCG dataset from Kaggle...")
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'adampq/pokemon-tcg-all-cards-1999-2023', '-p', temp_dir],
        capture_output=True,
        text=True
    )
    
    # Step 2: Extract the dataset
    zip_path = f'{temp_dir}/pokemon-tcg-all-cards-1999-2023.zip'
    if os.path.exists(zip_path):
        print("Extracting dataset...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(temp_dir)
    
    # Step 3: Load the CSV
    csv_files = [f for f in os.listdir(temp_dir) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No CSV files found in dataset directory")
    
    cards_file = f'{temp_dir}/{csv_files[0]}'
    df_raw = pd.read_csv(cards_file)
    print(f"Loaded {len(df_raw)} total cards from Kaggle dataset\n")
    
    # Step 4: Download images for subset
    df = df_raw.head(num_cards).copy()
    print(f"Processing {len(df)} cards...")
    
    successful = []
    failed = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Downloading images"):
        try:
            card_id = row['id']
            set_code, number = card_id.split('-')
            
            downloaded = False
            for url_template in [
                f"https://images.pokemontcg.io/{set_code}/{number}_hires.png",
                f"https://images.pokemontcg.io/{set_code}/{number}.png"
            ]:
                try:
                    response = requests.get(url_template, timeout=10)
                    response.raise_for_status()
                    
                    img = Image.open(BytesIO(response.content)).convert('RGB')
                    img_path = f"{save_dir}/images/card_{idx:05d}.jpg"
                    img.save(img_path, 'JPEG', quality=95)
                    
                    df.at[idx, 'image_path'] = img_path
                    successful.append(idx)
                    downloaded = True
                    break
                except:
                    continue
            
            if not downloaded:
                failed.append(card_id)
                
        except Exception as e:
            failed.append(row.get('id', f'idx_{idx}'))
            continue
    
    # Step 5: Save only the final processed dataset
    df_final = df.loc[successful].copy()
    csv_path = f'{save_dir}/pokemon_cards_metadata.csv'
    df_final.to_csv(csv_path, index=False)
    
    # Step 6: Clean up temporary files
    print("\nCleaning up temporary files...")
    import shutil
    shutil.rmtree(temp_dir)
    
    print(f"Successfully processed: {len(df_final)} cards")
    print(f"Failed to download: {len(failed)} cards")
    print(f"Images saved to: {save_dir}/images/")
    print(f"Metadata saved to: {csv_path}")
    
    return df_final

In [ ]:
save_dir = 'pokemon_cards'

# Download and process the dataset
df = download_and_process_pokemon_cards(save_dir, num_cards=1000)

print(f"Dataset shape: {df.shape}")
print(f"\nColumns ({len(df_radfw.columns)}):")
for col in df.columns:
    print(f"  - {col}")

print(f"\nFirst few rows:")
df.head()

### Load the dataset and preprocess it

In [11]:
# Load the processed data
cards_df = pd.read_csv('./pokemon_cards/pokemon_cards_metadata.csv')

print(f"Dataset shape: {cards_df.shape}")
print(f"\nColumns: {list(cards_df.columns)}")
cards_df.head()

Dataset shape: (1000, 30)

Columns: ['id', 'set', 'series', 'publisher', 'generation', 'release_date', 'artist', 'name', 'set_num', 'types', 'supertype', 'subtypes', 'level', 'hp', 'evolvesFrom', 'evolvesTo', 'abilities', 'attacks', 'weaknesses', 'retreatCost', 'convertedRetreatCost', 'rarity', 'flavorText', 'nationalPokedexNumbers', 'legalities', 'resistances', 'rules', 'regulationMark', 'ancientTrait', 'image_path']


,id,set,series,publisher,generation,release_date,artist,name,set_num,types,...,convertedRetreatCost,rarity,flavorText,nationalPokedexNumbers,legalities,resistances,rules,regulationMark,ancientTrait,image_path
0,base1-1,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Alakazam,1,['Psychic'],...,3.0,Rare Holo,Its brain can outperform a supercomputer. Its ...,[65],{'unlimited': 'Legal'},NaN,NaN,NaN,NaN,pokemon_cards/images/card_00000.jpg
1,base1-2,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Blastoise,2,['Water'],...,3.0,Rare Holo,A brutal Pokémon with pressurized water jets o...,[9],{'unlimited': 'Legal'},NaN,NaN,NaN,NaN,pokemon_cards/images/card_00001.jpg
2,base1-3,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Chansey,3,['Colorless'],...,1.0,Rare Holo,A rare and elusive Pokémon that is said to bri...,[113],{'unlimited': 'Legal'},"[{'type': 'Psychic', 'value': '-30'}]",NaN,NaN,NaN,pokemon_cards/images/card_00002.jpg
3,base1-4,Base,Base,WOTC,First,1/9/1999,Mitsuhiro Arita,Charizard,4,['Fire'],...,3.0,Rare Holo,Spits fire that is hot enough to melt boulders...,[6],{'unlimited': 'Legal'},"[{'type': 'Fighting', 'value': '-30'}]",NaN,NaN,NaN,pokemon_cards/images/card_00003.jpg
4,base1-5,Base,Base,WOTC,First,1/9/1999,Ken Sugimori,Clefairy,5,['Colorless'],...,1.0,Rare Holo,Its magical and cute appeal has many admirers....,[35],{'unlimited': 'Legal'},"[{'type': 'Psychic', 'value': '-30'}]",NaN,NaN,NaN,pokemon_cards/images/card_00004.jpg


In [18]:
import ast

def safe_parse_list(value):
    """Convert string representation of list to actual list."""
    if pd.isna(value):
        return None
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except:
            return None
    return value

# Parse list columns
list_columns = ['types', 'subtypes', 'retreatCost', 'nationalPokedexNumbers']
for col in list_columns:
    cards_df[f'{col}_parsed'] = cards_df[col].apply(safe_parse_list)

# Extract first type (most important)
cards_df['primary_type'] = cards_df['types_parsed'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

# Extract first subtype
cards_df['primary_subtype'] = cards_df['subtypes_parsed'].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None
)

# Convert retreat cost to numeric
cards_df['retreat_cost_count'] = cards_df['retreatCost_parsed'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print("Parsed list columns")
cards_df[['name', 'primary_type', 'primary_subtype', 'hp', 'rarity']].head(10)

Parsed list columns


,name,primary_type,primary_subtype,hp,rarity
0,Alakazam,Psychic,Stage 2,80.0,Rare Holo
1,Blastoise,Water,Stage 2,100.0,Rare Holo
2,Chansey,Colorless,Basic,120.0,Rare Holo
3,Charizard,Fire,Stage 2,120.0,Rare Holo
4,Clefairy,Colorless,Basic,40.0,Rare Holo
5,Gyarados,Water,Stage 1,100.0,Rare Holo
6,Hitmonchan,Fighting,Basic,70.0,Rare Holo
7,Machamp,Fighting,Stage 2,100.0,Rare Holo
8,Magneton,Lightning,Stage 1,60.0,Rare Holo
9,Mewtwo,Psychic,Basic,60.0,Rare Holo


In [20]:
# Convert HP to numeric (some might be strings)
cards_df['hp_numeric'] = pd.to_numeric(cards_df['hp'], errors='coerce')

# Count attacks
cards_df['num_attacks'] = cards_df['attacks'].apply(
    lambda x: len(safe_parse_list(x)) if pd.notna(x) else 0
)

# Boolean: has weakness/resistance
cards_df['has_weakness'] = cards_df['weaknesses'].notna()
cards_df['has_resistance'] = cards_df['resistances'].notna()

print("Created numeric features")
cards_df[['name', 'hp_numeric', 'num_attacks', 'retreat_cost_count', 'has_weakness']].head(10)

Created numeric features


,name,hp_numeric,num_attacks,retreat_cost_count,has_weakness
0,Alakazam,80.0,1,3,True
1,Blastoise,100.0,1,3,True
2,Chansey,120.0,2,1,True
3,Charizard,120.0,1,3,True
4,Clefairy,40.0,2,1,True
5,Gyarados,100.0,2,3,True
6,Hitmonchan,70.0,2,2,True
7,Machamp,100.0,1,3,True
8,Magneton,60.0,2,1,True
9,Mewtwo,60.0,2,3,True


In [21]:
# Keep only Pokemon cards (they have types and HP)
pokemon_only = cards_df[
    (cards_df['supertype'] == 'Pokémon') & 
    (cards_df['primary_type'].notna())
].copy()

print(f"Original: {len(cards_df)} cards")
print(f"Pokemon only: {len(pokemon_only)} cards")
print(f"\nType distribution:")
print(pokemon_only['primary_type'].value_counts())

Original: 1000 cards
Pokemon only: 800 cards

Type distribution:
primary_type
Grass        178
Colorless    140
Water        126
Psychic       96
Fighting      93
Lightning     75
Fire          70
Metal         11
Darkness      11
Name: count, dtype: int64


In [27]:
print(f"\nDataset size: {len(pokemon_only)} Pokemon cards")

print(f"\nMissing values in key columns:")
key_cols = ['primary_type', 'hp_numeric', 'rarity', 'artist', 'num_attacks']
for col in key_cols:
    missing = pokemon_only[col].isna().sum()
    pct = (missing / len(pokemon_only)) * 100
    print(f"  {col}: {missing} ({pct:.1f}%)")



Dataset size: 800 Pokemon cards

Missing values in key columns:
  primary_type: 0 (0.0%)
  hp_numeric: 0 (0.0%)
  rarity: 18 (2.2%)
  artist: 0 (0.0%)
  num_attacks: 0 (0.0%)
